# Домашнее задание  2 
RAG и память агента

In [1]:
%pip install -q requests pydantic python-dotenv pandas matplotlib numpy pymilvus pypdf

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [13]:
import os, re, sys, json, time, hashlib, threading
from pathlib import Path
from dataclasses import dataclass, field
from concurrent.futures import ThreadPoolExecutor
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pydantic import BaseModel, Field
from pymilvus import MilvusClient
from pypdf import PdfReader

from dotenv import load_dotenv
load_dotenv(dotenv_path=Path(".env"))
assert os.getenv("OPENROUTER_API_KEY"), ("Нет ключа OPENROUTER_API_KEY в .env")


CHAT_URL = "https://openrouter.ai/api/v1/chat/completions"
EMBED_URL = "https://openrouter.ai/api/v1/embeddings"
HEADERS = {"Authorization": f"Bearer {os.environ['OPENROUTER_API_KEY']}", "X-Title": "llm-rag-homework-02"}
MODELS = {"cheap": "openai/gpt-4o-mini", "mid": "anthropic/claude-haiku-4.5", "strong": "anthropic/claude-sonnet-4.6"}
EMBED_MODEL, DIM = "openai/text-embedding-3-small", 512
COLORS = {"violet": "#5436A3", "amber": "#F09000", "teal": "#00838F", "red": "#C43C3C", "grey": "#787882"}
DATA = Path("data")
PDF_DIR = DATA / "pdf"
MILVUS_URI = os.getenv("MILVUS_URI", "http://localhost:19530")
Path("img").mkdir(exist_ok=True)


@dataclass
class Ledger:
    calls: list = field(default_factory=list)
    local: threading.local = field(default_factory=threading.local)

    def add(self, tag, model, usage, seconds=0.0):
        cost = usage.get("cost") or 0.0
        self.calls.append({"tag": tag, "model": model.split("/")[-1], "prompt": usage.get("prompt_tokens", 0),
                           "completion": usage.get("completion_tokens", 0), "cost": cost, "seconds": round(seconds, 2)})
        self.local.spent = self.mine + cost
        return cost

    @property
    def total(self):
        return sum(c["cost"] for c in self.calls)

    @property
    def mine(self):
        return getattr(self.local, "spent", 0.0)

    def table(self):
        frame = pd.DataFrame(self.calls, columns=["tag", "model", "prompt", "completion", "cost", "seconds"])
        return frame.groupby(["tag", "model"]).agg(calls=("cost", "size"), prompt=("prompt", "sum"),
                                                                      completion=("completion", "sum"), cost=("cost", "sum")).round(5)


ledger = Ledger()


def post_with_retry(url, body, attempts=5):
    problem = "нет ответа"
    for attempt in range(attempts):
        try:
            r = requests.post(url, json=body, headers=HEADERS, timeout=120)
            if r.status_code == 200:
                return r.json()
            problem = f"HTTP {r.status_code}: {r.text[:200]}"
            if r.status_code not in (429, 500, 502, 503, 504):
                break
        except requests.RequestException as e:
            problem = type(e).__name__
        time.sleep(2.0 * (attempt + 1))
    raise RuntimeError(problem)


def chat(messages, model, tools=None, tag="chat"):
    body = {"model": model, "messages": messages, "temperature": 0, "usage": {"include": True}}
    if tools:
        body["tools"] = tools
    started = time.perf_counter()
    data = post_with_retry(CHAT_URL, body)
    ledger.add(tag, model, data.get("usage") or {}, time.perf_counter() - started)
    return data["choices"][0]["message"]


def load_jsonl(name):
    return [json.loads(line) for line in (DATA / name).read_text(encoding="utf-8").splitlines() if line.strip()]

In [9]:
print("Ключ OpenRouter найден:", bool(os.getenv("OPENROUTER_API_KEY")))

Ключ OpenRouter найден: True


In [14]:
print("Milvus:", MILVUS_URI)

Milvus: http://localhost:19530


#### Загрузка pdf

In [2]:
DATA = Path("data")
PDF_DIR = DATA / "pdf"

PDF_URLS = { "1707.07435v7.pdf": "https://arxiv.org/pdf/1707.07435", "1812.02353v3.pdf": "https://arxiv.org/pdf/1812.02353",
    "2302.09178v2.pdf": "https://arxiv.org/pdf/2302.09178", "5716.pdf": "https://storage.googleapis.com/gweb-research2023-media/pubtools/5716.pdf",
    "45530.pdf": "https://research.google.com/pubs/archive/45530.pdf",
    "A_Survey_of_Neural_Recommender_Systems.pdf": "https://le-wu.com/files/Publications/JOURNAL/A_Survey_of_Neural_Recommender_Systems.pdf",
    "youtube-multitask.pdf": "https://daiwk.github.io/assets/youtube-multitask.pdf",
    "youtube_mozilla_2021.pdf": "https://assets.mofoprod.net/network/documents/Mozilla_YouTube_Regrets_Report.pdf"}


In [3]:
pdf_pages = []

for path in sorted(PDF_DIR.glob("*.pdf")):
    reader = PdfReader(str(path))

    for page_number, page in enumerate(reader.pages, start=1):
        text = page.extract_text() or ""

        if not text.strip():
            continue

        pdf_pages.append({ "page": path.name, "page_number": page_number,"section": f"Page {page_number}","url": PDF_URLS[path.name],"text": text })

print("Загружено страниц:", len(pdf_pages))
print("Всего символов:", sum(len(p["text"]) for p in pdf_pages))

could not convert string to float: b'0.00-34120734' : FloatObject (b'0.00-34120734') invalid; use 0.0 instead


Загружено страниц: 142
Всего символов: 624004


#### Нарезка документов  
1)Наивная нарезка (char_chunks) - каждые 400 символов, даже если предложение обрывается  
2)Структурная нарезка (structured_chunks) - объединяем целые предложения, пока их суммарная длина не приблизится к 400 символам. Если следующее предложение не помещается, начинаем новый чанк.  
  
Каждый фрагмент сохраняет метаданные исходного документа:название PDF, номер страницы и ссылку на источник.

**Наивная нарезка:**

In [5]:
def chunk_chars(page, size=400):
    text = " ".join(page["text"].split())
    return [{ **page, "text": text[i:i + size]} for i in range(0, len(text), size)]

char_chunks = [ chunk 
                for page in pdf_pages 
                for chunk in chunk_chars(page)]

print("Количество фрагментов:", len(char_chunks))
print("Размер первого фрагмента:", len(char_chunks[0]["text"]))
print("\nПример фрагмента:\n")
print(char_chunks[0]["text"])

Количество фрагментов: 1608
Размер первого фрагмента: 400

Пример фрагмента:

1 Deep Learning based Recommender System: A Survey and New Perspectives SHUAI ZHANG, University of New South Wales LINA YAO, University of New South Wales AIXIN SUN, Nanyang Technological University YI TAY, Nanyang Technological University With the ever-growing volume of online information, recommender systems have been an eﬀective strategy to overcome such information overload. /T_he utility of r


**Структурная нарезка:**

In [6]:
def chunk_structured(page, size=400):
    text = page["text"]
    text = re.sub( r"([A-Za-z])-\s*\n\s*([a-z])",  r"\1\2", text)# восстанавливаем слова, разделенные переносом строки
    text = " ".join(text.split())
    sentences = re.split( r"(?<=[.!?])\s+(?=[A-Z0-9])", text)

    units = []

    for sentence in sentences:
        words = sentence.split()
        part = ""

        for word in words:
            if part and len(part) + 1 + len(word) > size:
                units.append(part)
                part = word
            else:
                part = f"{part} {word}".strip()

        if part:
            units.append(part)

    chunks = []
    current = ""

    for unit in units:
        if current and len(current) + 1 + len(unit) > size:
            chunks.append({**page, "text": current })
            current = unit
        else:
            current = f"{current} {unit}".strip()

    if current:
        chunks.append({  **page, "text": current })
    return chunks


structured_chunks = [ chunk
                      for page in pdf_pages
                        for chunk in chunk_structured(page)]

print("Количество фрагментов:", len(structured_chunks))
print("Размер первого фрагмента:", len(structured_chunks[0]["text"]))
print("\nПример фрагмента:\n")
print(structured_chunks[0]["text"])

Количество фрагментов: 1940
Размер первого фрагмента: 398

Пример фрагмента:

1 Deep Learning based Recommender System: A Survey and New Perspectives SHUAI ZHANG, University of New South Wales LINA YAO, University of New South Wales AIXIN SUN, Nanyang Technological University YI TAY, Nanyang Technological University With the ever-growing volume of online information, recommender systems have been an eﬀective strategy to overcome such information overload. /T_he utility of


In [7]:
comparison = pd.DataFrame({
    "Метод": [ "Наивная", "Структурная"],
    "Количество чанков": [  len(char_chunks),len(structured_chunks)],
    "Средняя длина": [np.mean([len(c["text"]) for c in char_chunks]), np.mean([len(c["text"]) for c in structured_chunks])],
    "Минимальная длина": [ min(len(c["text"]) for c in char_chunks), min(len(c["text"]) for c in structured_chunks) ],
    "Максимальная длина": [ max(len(c["text"]) for c in char_chunks), max(len(c["text"]) for c in structured_chunks)]})

comparison.round(1)

,Метод,Количество чанков,Средняя длина,Минимальная длина,Максимальная длина
0,Наивная,1608,382.1,1,400
1,Структурная,1940,314.8,19,400


In [12]:
assert len(char_chunks) >= 300
assert len(structured_chunks) >= 300